做了什麼事情：


1.   星期一到五全部的課表都有
2.   有氣溫資料
3.   電表資料用的是永續辦給的總表



In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
import statsmodels.api as sm

sns.set(style="whitegrid")
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']

In [50]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib as mpl
import matplotlib.font_manager as fm

!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

fm.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
mpl.rc('font', family='Taipei Sans TC Beta')

--2026-04-21 17:18:16--  https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_
Resolving drive.google.com (drive.google.com)... 173.194.206.113, 173.194.206.138, 173.194.206.139, ...
Connecting to drive.google.com (drive.google.com)|173.194.206.113|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ [following]
--2026-04-21 17:18:16--  https://drive.usercontent.google.com/download?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.132.132, 2607:f8b0:4001:c00::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.132.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20659344 (20M) [application/octet-stream]
Saving to: ‘TaipeiSansTCBeta-Regular.ttf’

TaipeiSansTCBeta-Re 100%[===================>]  19.70M  56.1MB/s    in 0.4s    

2026-04-21 17

In [51]:
# 1. 載入電力數據 (直接讀取 Excel)
ele_df = pd.read_excel('2025_ele_all_buildings_V3.xlsx', sheet_name=0)

# 2. 載入課表與天氣 CSV
schedule_df = pd.read_csv('課表資料.csv')
weather_df = pd.read_csv('466920_2025.csv')

# --- 電力數據前處理 ---
ele_df['DateTime'] = pd.to_datetime(ele_df['DateTime'])
# 篩選分析區間
mask = (ele_df['DateTime'] >= '2025-03-01') & (ele_df['DateTime'] <= '2025-06-10 23:59:59')
df = ele_df[mask].copy()
df['DoW'] = df['DateTime'].dt.dayofweek + 1 # 周一=1
df['Hour_Num'] = df['DateTime'].dt.hour
df['Hour_Str'] = df['DateTime'].dt.strftime('%H:00')

# --- 天氣數據前處理 (Tx 為氣溫) ---
weather_df = weather_df.rename(columns={'Unnamed: 0': 'DateTime', 'Tx': 'Temp'})
weather_df['DateTime'] = pd.to_datetime(weather_df['DateTime'])
weather_df = weather_df[['DateTime', 'Temp']]

# --- 合併資料 ---
# 合併氣溫
merged_df = pd.merge(df, weather_df, on='DateTime', how='left')
merged_df['Temp'] = merged_df['Temp'].interpolate() # 填補缺失氣溫

# 合併課表
schedule_df = schedule_df[['DoW', 'Time', 'BigC', 'MediumC', 'SmallC']].dropna(subset=['Time'])
merged_df = pd.merge(merged_df, schedule_df, left_on=['DoW', 'Hour_Str'], right_on=['DoW', 'Time'], how='left')

# 補齊非課程時段
merged_df[['BigC', 'MediumC', 'SmallC']] = merged_df[['BigC', 'MediumC', 'SmallC']].fillna(0)
merged_df = merged_df.dropna(subset=['AT1009'])

print(f"資料整合成功，共有 {len(merged_df)} 筆小時能耗數據。")

資料整合成功，共有 2448 筆小時能耗數據。


In [52]:
# 建立小時固定效應 (捕捉大樓日常作息)
hour_dummies = pd.get_dummies(merged_df['Hour_Num'], prefix='H').astype(int)

# 設定特徵變數：教室數量 + 氣溫 + 小時效應
X = pd.concat([merged_df[['BigC', 'MediumC', 'SmallC', 'Temp']], hour_dummies], axis=1)
y = merged_df['AT1009']

model = LinearRegression()
model.fit(X, y)

# 關鍵參數提取
bigc_coef = model.coef_[0] # 每間大教室每小時耗電
temp_coef = model.coef_[3] # 氣溫每升一度增加的能耗
r2_score = model.score(X, y)
total_bigc_hours = merged_df['BigC'].sum()
total_consumption = bigc_coef * total_bigc_hours

print("【分析結果摘要】")
print(f"1. 模型解釋力 (R-squared): {r2_score:.4f}")
print(f"2. 大教室 (BigC) 單位耗電量: {bigc_coef:.2f} kWh/hr")
print(f"3. 氣溫對基礎電力影響: {temp_coef:.2f} kWh/°C")
print(f"4. 大教室期間預估總耗電量: {total_consumption:.2f} kWh")

【分析結果摘要】
1. 模型解釋力 (R-squared): 0.7906
2. 大教室 (BigC) 單位耗電量: 5.82 kWh/hr
3. 氣溫對基礎電力影響: 0.06 kWh/°C
4. 大教室期間預估總耗電量: 7416.75 kWh


In [53]:
# 列出所有欄位名稱與索引，看清楚 1,2,3,4,6,7 到底是什麼
for i, col in enumerate(df.columns):
    print(f"索引 {i}: 欄位名稱 {col}")

索引 0: 欄位名稱 DateTime
索引 1: 欄位名稱 AT1009
索引 2: 欄位名稱 DoW
索引 3: 欄位名稱 Hour_Num
索引 4: 欄位名稱 Hour_Str


In [54]:
features = ['BigC', 'MediumC', 'SmallC', 'Temp'] # 加上你想要的氣象欄位名稱
X = merged_df[features]
X = sm.add_constant(X)
y = merged_df['AT1009']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 AT1009   R-squared:                       0.610
Model:                            OLS   Adj. R-squared:                  0.609
Method:                 Least Squares   F-statistic:                     954.5
Date:                Tue, 21 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:18:18   Log-Likelihood:                -9826.8
No. Observations:                2448   AIC:                         1.966e+04
Df Residuals:                    2443   BIC:                         1.969e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.9462      1.122      5.299      0.0

# 先計算基礎用電（假日）再算平日增加了多少

In [55]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 1. 資料讀取與基本清理
# ==========================================
ele_df = pd.read_excel('2025_ele_all_buildings_V3.xlsx')
ele_df['DateTime'] = pd.to_datetime(ele_df['DateTime'])
weather_df = pd.read_csv('466920_2025.csv').rename(columns={'Unnamed: 0': 'DateTime', 'Tx': 'Temp'})
weather_df['DateTime'] = pd.to_datetime(weather_df['DateTime'])
schedule_df = pd.read_csv('課表資料.csv')

# 合併電力與天氣
df = pd.merge(ele_df, weather_df[['DateTime', 'Temp']], on='DateTime', how='left')
df['Temp'] = df['Temp'].interpolate()
df['DoW'] = df['DateTime'].dt.dayofweek + 1 # 1=Mon, 7=Sun
df['Hour'] = df['DateTime'].dt.strftime('%H:00')

# 2. 建立假日基準線模型 (Baseline)
# ==========================================
# 篩選週六(6)與週日(7)作為假日基準
holiday_df = df[df['DoW'].isin([6, 7])].dropna(subset=['AT1009', 'Temp'])

# 建立假日的「氣溫 vs 耗電」模型
# 這裡使用簡單線性回歸，找出假日底噪隨氣溫變化的規律
X_base = sm.add_constant(holiday_df['Temp'])
base_model = sm.OLS(holiday_df['AT1009'], X_base).fit()

# 3. 計算平日的「多餘耗電量」
# ==========================================
# 篩選分析區間與平日
mask = (df['DateTime'] >= '2025-03-01') & (df['DateTime'] <= '2025-06-10 23:59:59')
work_df = df[mask].copy()

# 預測在當前氣溫下的「理想底噪」
X_work_base = sm.add_constant(work_df['Temp'])
work_df['Estimated_Base'] = base_model.predict(X_work_base)

# 計算差額：實際電力 - 假日基準底噪 = 課程活動造成的電力 (Excess_Energy)
work_df['Excess_Energy'] = work_df['AT1009'] - work_df['Estimated_Base']

# 合併課表資料
schedule_clean = schedule_df[['DoW', 'Time', 'BigC', 'MediumC', 'SmallC']].dropna(subset=['Time'])
final_df = pd.merge(work_df, schedule_clean, left_on=['DoW', 'Hour'], right_on=['DoW', 'Time'], how='left')
final_df[['BigC', 'MediumC', 'SmallC']] = final_df[['BigC', 'MediumC', 'SmallC']].fillna(0)

# 4. 對「多餘耗電量」進行回歸分析
# ==========================================
# 此時的 y 是 Excess_Energy，自變數只需教室數量
# 這裡不需要 const，因為理論上沒課時 Excess_Energy 應為 0
X_final = final_df[['BigC', 'MediumC', 'SmallC']]
y_final = final_df['Excess_Energy']

final_regression = sm.OLS(y_final, X_final).fit()

# 5. 輸出結果
# ==========================================
print("【第一階段：假日底噪模型摘要】")
print(f"假日底噪公式: Power = {base_model.params[0]:.2f} + {base_model.params[1]:.2f} * Temp")
print("-" * 50)
print("【第二階段：教室加權回歸結果 (針對多餘耗電)】")
print(final_regression.summary())

# 計算大教室總耗電
bigc_unit_energy = final_regression.params['BigC']
total_bigc_energy = final_df['BigC'].sum() * bigc_unit_energy
print(f"\n基於基準線法的大教室單位能耗: {bigc_unit_energy:.4f} kWh/hr")
print(f"大教室期間預估總耗電量: {total_bigc_energy:.2f} kWh")

【第一階段：假日底噪模型摘要】
假日底噪公式: Power = 15.21 + 0.05 * Temp
--------------------------------------------------
【第二階段：教室加權回歸結果 (針對多餘耗電)】
                                 OLS Regression Results                                
Dep. Variable:          Excess_Energy   R-squared (uncentered):                   0.623
Model:                            OLS   Adj. R-squared (uncentered):              0.622
Method:                 Least Squares   F-statistic:                              1347.
Date:                Tue, 21 Apr 2026   Prob (F-statistic):                        0.00
Time:                        17:18:19   Log-Likelihood:                         -9915.1
No. Observations:                2448   AIC:                                  1.984e+04
Df Residuals:                    2445   BIC:                                  1.985e+04
Df Model:                           3                                                  
Covariance Type:            nonrobust                                           

/tmp/ipykernel_1955/2519749954.py:60: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"假日底噪公式: Power = {base_model.params[0]:.2f} + {base_model.params[1]:.2f} * Temp")
